# sparse_release_cap10_rebound_balance_v1

Notebook-first diagnostic after `sparse_release_drawdown_target_tightening_v1` found that `cap=0.10` passes the two evaluated drawdown gates but looks close to a cash-defense solution. This notebook keeps `0.10` as the downtrend risk anchor, then selectively lifts only sparse release/rebound rows from `sparse_rebound_release_target_revision_v1`.

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_release_drawdown_target_tightening_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection` / training target design / BC sample weighting.
- **Target windows**: `wf_2022` held out in 2022; `wf_2023_2024` held out in 2023-2024.
- **Accepted trade-off**: keep +30pp relative return and <=15% MaxDD while improving mean exposure versus the pure `cap=0.10` anchor.
- **Failure mode**: release participation lift breaches drawdown, or the only passing target remains underparticipated.

External basis retained for this design:

- Qlib experiment/replay framework: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO caution: clipped policy optimization can drift from a useful initialization; PPO is not part of this screen: https://arxiv.org/abs/1707.06347
- Learning from demonstrations / BC warm start before RL: https://arxiv.org/abs/1704.03732
- Backtest overfitting control / CSCV discipline: https://doi.org/10.21314/JCF.2016.322


In [1]:

from __future__ import annotations

import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sparse_release_cap10_rebound_balance_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/sparse_rebound_release_target_revision_v1'
DIRECT_COMPARISON_CARD = ROOT / 'runtime/model_selection/card/sparse_release_drawdown_target_tightening_v1'
TARGETS_PATH = SOURCE_CARD / 'sparse_release_revised_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
RISK_ANCHOR_CAP = 0.10
RELEASE_CAP_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
MIN_MEAN_AVG_POS = 0.13
MIN_RELEASE_SLICE_ACCURACY = 0.55
MIN_RELEASE_SAMPLES = 20
MIN_DOWNTREND_SLICE_ACCURACY = 0.60
MIN_DOWNTREND_SAMPLES = 100
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'source_target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
        'target_train': ('2020-01-02', '2021-12-31'),
    },
    'wf_2023_2024': {
        'source_target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2021-12-31'),
        'valid': ('2022-01-01', '2022-12-31'),
        'test': ('2023-01-01', '2024-12-31'),
        'target_train': ('2020-01-02', '2022-12-30'),
    },
}

SCOPE_FOLDS = [
    {
        'name': 'wf_2022_train',
        'train_start': '2020-01-02',
        'train_end': '2021-12-31',
        'eval_start': '2022-01-03',
        'eval_end': '2022-12-30',
    },
    {
        'name': 'wf_2023_2024_train',
        'train_start': '2020-01-02',
        'train_end': '2022-12-30',
        'eval_start': '2023-01-03',
        'eval_end': '2024-12-31',
    },
]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def as_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.map(lambda v: False if pd.isna(v) else str(v).strip().lower() in {'true', '1', 'yes', 'y'}).astype(bool)


def clean_target_frame(targets: pd.DataFrame) -> pd.DataFrame:
    frame = targets.copy()
    frame['datetime'] = pd.to_datetime(frame['datetime']).dt.normalize()
    for col in ['teacher_position', 'rebound_target_position', 'risk_target_position', 'objective_weight']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').fillna(0.0)
    for col in ['is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'release_gate_predicted']:
        if col in frame.columns:
            frame[col] = as_bool(frame[col])
    if 'release_gate_state' not in frame.columns:
        frame['release_gate_state'] = False
    return frame


def build_release_cap_targets(base_targets: pd.DataFrame, *, release_cap: float) -> pd.DataFrame:
    targets = clean_target_frame(base_targets)
    targets['source_risk_target_position'] = targets['risk_target_position']
    if 'risk_target_position_original' in targets.columns:
        targets['source_risk_target_position_original'] = pd.to_numeric(
            targets['risk_target_position_original'], errors='coerce'
        ).fillna(targets['risk_target_position'])
    else:
        targets['source_risk_target_position_original'] = targets['risk_target_position']
    targets['risk_target_position_original'] = float(RISK_ANCHOR_CAP)
    targets['risk_anchor_cap'] = float(RISK_ANCHOR_CAP)
    targets['release_cap'] = float(release_cap)
    targets['target_variant'] = f'cap10_release_{release_cap:.2f}'
    downtrend = targets['is_downtrend_state'].astype(bool)
    release = targets['release_gate_state'].astype(bool) & downtrend
    targets.loc[downtrend, 'risk_target_position'] = float(RISK_ANCHOR_CAP)
    release_position = np.maximum(
        float(RISK_ANCHOR_CAP),
        np.minimum(targets['rebound_target_position'].to_numpy(dtype=float), float(release_cap)),
    )
    targets.loc[release, 'risk_target_position'] = release_position[release.to_numpy()]
    targets['release_cap_effective_lift'] = targets['risk_target_position'] - targets['risk_target_position_original']
    targets['release_cap_active'] = (release & (targets['release_cap_effective_lift'] > 1e-9)).astype(bool)
    targets['target_semantics_revision'] = np.where(
        targets['release_cap_active'],
        'cap10_sparse_release_lift',
        np.where(downtrend, 'cap10_downtrend_anchor', ''),
    )
    return targets.reset_index(drop=True)


def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    out['backtest_return'] = float(bt.get('return', out.get('return', 0.0)))
    out['backtest_max_dd'] = float(bt.get('max_dd', out.get('max_dd', 0.0)))
    out['backtest_relative_return'] = float(bt.get('relative_outperformance', out.get('relative_return', 0.0)))
    return out


def prefixed_metrics(prefix: str, metrics: dict) -> dict[str, float]:
    keys = ['return', 'sharpe', 'max_dd', 'avg_pos', 'stress_avg_pos', 'buy_and_hold_return', 'relative_return']
    return {f'{prefix}_{k}': float(metrics[k]) for k in keys if k in metrics}


def slice_gate_summary(slice_metrics: pd.DataFrame) -> dict[str, object]:
    if slice_metrics.empty:
        return {
            'slice_fit_gate': False,
            'release_slice_gate': False,
            'downtrend_slice_gate': False,
            'release_samples': 0,
            'release_accuracy': None,
            'downtrend_samples': 0,
            'downtrend_accuracy': None,
        }
    metrics = slice_metrics.set_index('slice')

    def row_values(name: str) -> tuple[int, float]:
        if name not in metrics.index:
            return 0, float('nan')
        row = metrics.loc[name]
        return int(row.get('samples', 0)), float(row.get('accuracy', float('nan')))

    release_samples, release_accuracy = row_values('release_gate_state')
    downtrend_samples, downtrend_accuracy = row_values('is_downtrend_state')
    release_gate = bool(
        release_samples >= MIN_RELEASE_SAMPLES
        and np.isfinite(release_accuracy)
        and release_accuracy >= MIN_RELEASE_SLICE_ACCURACY
    )
    downtrend_gate = bool(
        downtrend_samples >= MIN_DOWNTREND_SAMPLES
        and np.isfinite(downtrend_accuracy)
        and downtrend_accuracy >= MIN_DOWNTREND_SLICE_ACCURACY
    )
    return {
        'slice_fit_gate': bool(release_gate and downtrend_gate),
        'release_slice_gate': release_gate,
        'downtrend_slice_gate': downtrend_gate,
        'release_samples': release_samples,
        'release_accuracy': release_accuracy if np.isfinite(release_accuracy) else None,
        'downtrend_samples': downtrend_samples,
        'downtrend_accuracy': downtrend_accuracy if np.isfinite(downtrend_accuracy) else None,
    }


def target_score(relative_return: float, max_dd: float, avg_pos: float, slice_gate: bool) -> float:
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN - float(relative_return))
    drawdown_gap = max(0.0, float(max_dd) - TARGET_MAX_DD)
    participation_gap = max(0.0, MIN_MEAN_AVG_POS - float(avg_pos)) * 20.0
    slice_penalty = 5.0 if not slice_gate else 0.0
    return relative_gap + drawdown_gap * 4.0 + participation_gap + slice_penalty


In [3]:

profile = load_profile(PROFILE_NAME)
source_selection = json.loads((SOURCE_CARD / 'selection.json').read_text(encoding='utf-8'))
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
direct_selection = json.loads((DIRECT_COMPARISON_CARD / 'selection.json').read_text(encoding='utf-8'))
direct_summary = json.loads((DIRECT_COMPARISON_CARD / 'summary.json').read_text(encoding='utf-8'))
base_targets = clean_target_frame(pd.read_csv(TARGETS_PATH))
frame = pd.read_pickle(FRAME_PATH)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    fold_targets = base_targets[base_targets['fold'].astype(str).eq(spec['source_target_fold'])].copy()
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    eval_overlap = fold_targets[(fold_targets['datetime'] >= test_start) & (fold_targets['datetime'] <= test_end)]
    coverage_rows.append({
        'fold': fold_name,
        'source_target_fold': spec['source_target_fold'],
        'target_rows': int(len(fold_targets)),
        'target_start': str(fold_targets['datetime'].min().date()) if len(fold_targets) else '',
        'target_end': str(fold_targets['datetime'].max().date()) if len(fold_targets) else '',
        'release_gate_rows': int(as_bool(fold_targets['release_gate_state']).sum()) if len(fold_targets) else 0,
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'eval_overlap_rows': int(len(eval_overlap)),
        'fold_local_gate': bool(len(fold_targets) > 0 and len(eval_overlap) == 0),
    })
coverage = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_experiment': 'sparse_rebound_release_target_revision_v1',
    'source_decision': source_selection.get('decision'),
    'direct_comparison_experiment': 'sparse_release_drawdown_target_tightening_v1',
    'direct_comparison_decision': direct_selection.get('decision'),
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'risk_anchor_cap': RISK_ANCHOR_CAP,
    'release_cap_grid': RELEASE_CAP_GRID,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'min_mean_avg_pos': MIN_MEAN_AVG_POS,
    'windows': coverage_rows,
}
pd.DataFrame(coverage_rows).to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', coverage)
display(pd.DataFrame(coverage_rows))


,fold,source_target_fold,target_rows,target_start,target_end,release_gate_rows,train_start,train_end,valid_start,valid_end,test_start,test_end,eval_overlap_rows,fold_local_gate
0,wf_2022,wf_2022_train,505,2020-01-02,2021-12-31,40,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,0,True
1,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,40,2010-03-11,2021-12-31,2022-01-01,2022-12-31,2023-01-01,2024-12-31,0,True


In [4]:

variant_frames = []
scope_frames = []
lift_frames = []
variant_audit_rows = []
for release_cap in RELEASE_CAP_GRID:
    targets = build_release_cap_targets(base_targets, release_cap=release_cap)
    variant_frames.append(targets)
    scope_audit = audit_state_objective_target_scope(targets, folds=SCOPE_FOLDS)
    lift_audit = audit_state_objective_target_lift(
        targets,
        group_cols=('target_variant', 'fold'),
        baseline_risk_position_col='risk_target_position_original',
    )
    scope_audit.insert(0, 'target_variant', f'cap10_release_{release_cap:.2f}')
    lift_audit.insert(0, 'release_cap', float(release_cap))
    scope_frames.append(scope_audit)
    lift_frames.append(lift_audit)
    scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
    lift_summary = summarize_state_objective_target_lift_audit(lift_audit)
    variant_audit_rows.append({
        'target_variant': f'cap10_release_{release_cap:.2f}',
        'release_cap': float(release_cap),
        'target_rows': int(len(targets)),
        'release_gate_rows': int(targets['release_gate_state'].sum()),
        'active_release_lift_rows': int(targets['release_cap_active'].sum()),
        'mean_release_lift': float(targets.loc[targets['release_cap_active'], 'release_cap_effective_lift'].mean()) if targets['release_cap_active'].any() else 0.0,
        'scope_gate': bool(scope_summary['training_allowed']),
        'scope_eval_overlap_rows': int(scope_summary['eval_overlap_rows']),
        'target_lift_gate': bool(lift_summary['training_allowed']),
        'current_positive_rebound_lift_rows': int(lift_summary['current_positive_rebound_lift_rows']),
        'candidate_release_positive_lift_rows': int(lift_summary['candidate_release_positive_lift_rows']),
    })

all_variant_targets = pd.concat(variant_frames, ignore_index=True)
target_scope_audit = pd.concat(scope_frames, ignore_index=True) if scope_frames else pd.DataFrame()
target_lift_audit = pd.concat(lift_frames, ignore_index=True) if lift_frames else pd.DataFrame()
target_variant_coverage = pd.DataFrame(variant_audit_rows)

all_variant_targets.to_csv(CARD / 'selected_targets_by_variant.csv', index=False)
target_scope_audit.to_csv(CARD / 'target_scope_audit.csv', index=False)
target_lift_audit.to_csv(CARD / 'target_lift_audit.csv', index=False)
target_variant_coverage.to_csv(CARD / 'target_variant_coverage.csv', index=False)
write_json(CARD / 'target_scope_summary.json', summarize_state_objective_target_scope_audit(target_scope_audit))
write_json(CARD / 'target_lift_summary.json', summarize_state_objective_target_lift_audit(target_lift_audit))
display(target_variant_coverage)


,target_variant,release_cap,target_rows,release_gate_rows,active_release_lift_rows,mean_release_lift,scope_gate,scope_eval_overlap_rows,target_lift_gate,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows
0,cap10_release_0.10,0.10,1261,80,0,0.00,True,0,False,0,114
1,cap10_release_0.15,0.15,1261,80,80,0.05,True,0,True,80,114
2,cap10_release_0.20,0.20,1261,80,80,0.10,True,0,True,80,114
3,cap10_release_0.25,0.25,1261,80,80,0.15,True,0,True,80,114
4,cap10_release_0.30,0.30,1261,80,80,0.20,True,0,True,80,114
5,cap10_release_0.35,0.35,1261,80,80,0.25,True,0,True,80,114


In [5]:

init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=220, lr=0.001, batch_size=128, seed=SEED)
weight_config = StateObjectiveBCSampleWeightConfig(
    downtrend_multiplier=2.0,
    rebound_multiplier=2.0,
    ordinary_multiplier=1.0,
    objective_weight_power=0.5,
    class_balanced_beta=0.99,
    weight_cap=20.0,
)

fold_datasets = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    fold_datasets[fold_name] = {
        'dataset': dataset,
        'test_df': segment_frame(dataset, 'test'),
        'test_price': price_frame.loc[spec['test'][0]:spec['test'][1]],
    }

fold_rows = []
slice_frames = []
action_frames = []
train_metric_rows = []

for variant_idx, release_cap in enumerate(RELEASE_CAP_GRID):
    variant_name = f'cap10_release_{release_cap:.2f}'
    variant_targets = all_variant_targets[all_variant_targets['target_variant'].astype(str).eq(variant_name)].copy()
    for fold_idx, (fold_name, spec) in enumerate(WALK_FORWARD_WINDOWS.items()):
        fold_targets = variant_targets[variant_targets['fold'].astype(str).eq(spec['source_target_fold'])].copy()
        seed = SEED + variant_idx * 100 + fold_idx
        set_seed(seed)
        bc_result = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=StateObjectiveBCTrainingConfig(
                steps=train_config.steps,
                lr=train_config.lr,
                batch_size=train_config.batch_size,
                seed=seed,
            ),
            sample_weight_config=weight_config,
            sample_size=0,
            rng_seed=seed,
        )
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=seed,
        )
        slice_metrics = evaluate_bc_actor_slice_fit(
            bc_result.actor,
            bc_result.obs,
            bc_result.actions,
            bc_dataset.metadata,
            slice_cols=('release_gate_state', 'is_downtrend_state', 'is_rebound_cluster'),
        )
        slice_summary = slice_gate_summary(slice_metrics)
        tagged_slices = slice_metrics.copy()
        tagged_slices.insert(0, 'fold', fold_name)
        tagged_slices.insert(0, 'release_cap', float(release_cap))
        tagged_slices.insert(0, 'target_variant', variant_name)
        slice_frames.append(tagged_slices)
        action_dist = bc_result.action_distribution.copy()
        action_dist.insert(0, 'fold', fold_name)
        action_dist.insert(0, 'release_cap', float(release_cap))
        action_dist.insert(0, 'target_variant', variant_name)
        action_frames.append(action_dist)
        train_metric_rows.append({
            'target_variant': variant_name,
            'release_cap': float(release_cap),
            'fold': fold_name,
            'target_rows': int(len(fold_targets)),
            'release_gate_rows': int(fold_targets['release_gate_state'].sum()),
            'active_release_lift_rows': int(fold_targets['release_cap_active'].sum()),
            **{f'bc_{k}': v for k, v in bc_result.metrics.items()},
            **slice_summary,
        })
        actor = bc_result.actor
        actor.eval()
        data = fold_datasets[fold_name]
        metrics = evaluate_actor_path(actor, data['test_df'], data['test_price'], profile, spec['test'])
        relative_return = float(metrics.get('relative_return', 0.0))
        max_dd = float(metrics.get('max_dd', 0.0))
        avg_pos = float(metrics.get('avg_pos', 0.0))
        fold_gate = bool(relative_return >= TARGET_RELATIVE_RETURN and max_dd <= TARGET_MAX_DD)
        row = {
            'target_variant': variant_name,
            'release_cap': float(release_cap),
            'fold': fold_name,
            'status': 'evaluated',
            'seed': int(seed),
            'target_rows': int(len(fold_targets)),
            'release_gate_rows': int(fold_targets['release_gate_state'].sum()),
            'active_release_lift_rows': int(fold_targets['release_cap_active'].sum()),
            'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
            'bc_after_loss': float(bc_result.metrics['after_loss']),
            'bc_weight_max': float(bc_result.metrics.get('weight_max', float('nan'))),
            **slice_summary,
            'fold_gate': fold_gate,
            'target_score': target_score(relative_return, max_dd, avg_pos, bool(slice_summary['slice_fit_gate'])),
            **prefixed_metrics('test', metrics),
        }
        fold_rows.append(row)

fold_results = pd.DataFrame(fold_rows)
train_metrics = pd.DataFrame(train_metric_rows)
bc_slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()

summary_rows = []
for variant_name, group in fold_results.groupby('target_variant', sort=True):
    summary_rows.append({
        'target_variant': variant_name,
        'release_cap': float(group['release_cap'].iloc[0]),
        'folds_evaluated': int(len(group)),
        'fold_gate_pass_count': int(group['fold_gate'].sum()),
        'all_fold_gate': bool(group['fold_gate'].all()),
        'slice_gate_pass_count': int(group['slice_fit_gate'].sum()),
        'all_slice_gate': bool(group['slice_fit_gate'].all()),
        'participation_gate': bool(float(group['test_avg_pos'].mean()) >= MIN_MEAN_AVG_POS),
        'mean_target_score': float(group['target_score'].mean()),
        'worst_target_score': float(group['target_score'].max()),
        'mean_relative_return': float(group['test_relative_return'].mean()),
        'worst_max_dd': float(group['test_max_dd'].max()),
        'mean_avg_pos': float(group['test_avg_pos'].mean()),
        'min_release_accuracy': float(pd.to_numeric(group['release_accuracy'], errors='coerce').min()),
        'min_downtrend_accuracy': float(pd.to_numeric(group['downtrend_accuracy'], errors='coerce').min()),
    })

candidate_summary = pd.DataFrame(summary_rows)
if not candidate_summary.empty:
    candidate_summary['strict_gate'] = (
        candidate_summary['all_fold_gate'].astype(bool)
        & candidate_summary['all_slice_gate'].astype(bool)
        & candidate_summary['participation_gate'].astype(bool)
    )
    candidate_summary = candidate_summary.sort_values(
        ['strict_gate', 'all_fold_gate', 'mean_target_score', 'worst_max_dd', 'mean_avg_pos'],
        ascending=[False, False, True, True, False],
    ).reset_index(drop=True)

candidate_summary.to_csv(CARD / 'target_variant_summary.csv', index=False)
fold_results.to_csv(CARD / 'target_variant_fold_results.csv', index=False)
train_metrics.to_csv(CARD / 'bc_train_metrics.csv', index=False)
bc_slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'action_distribution.csv', index=False)

display(candidate_summary)
display(fold_results)


[55330:MainThread](2026-05-26 05:29:07,771) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55330:MainThread](2026-05-26 05:29:08,622) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55330:MainThread](2026-05-26 05:29:08,623) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[55330:MainThread](2026-05-26 05:29:10,667) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[55330:MainThread](2026-05-26 05:29:10,669) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,target_variant,release_cap,folds_evaluated,fold_gate_pass_count,all_fold_gate,slice_gate_pass_count,all_slice_gate,participation_gate,mean_target_score,worst_target_score,mean_relative_return,worst_max_dd,mean_avg_pos,min_release_accuracy,min_downtrend_accuracy,strict_gate
0,cap10_release_0.10,0.10,2,1,False,2,True,True,24.692751,49.385503,26.575972,12.291345,0.150793,1.000,0.968553,False
1,cap10_release_0.15,0.15,2,1,False,2,True,True,25.201626,50.403253,25.766610,13.470361,0.142767,0.975,0.952991,False
2,cap10_release_0.20,0.20,2,1,False,1,False,True,27.489779,54.979558,24.065299,11.401851,0.143395,0.025,0.872117,False
3,cap10_release_0.35,0.35,2,1,False,1,False,False,27.835320,54.670640,24.937429,13.631561,0.104374,0.000,0.901468,False
4,cap10_release_0.30,0.30,2,1,False,1,False,False,28.251784,55.848306,25.794454,12.291345,0.112959,0.000,0.884615,False
5,cap10_release_0.25,0.25,2,0,False,1,False,True,50.756001,54.287107,17.968874,26.806224,0.145686,0.000,0.893162,False


,target_variant,release_cap,fold,status,seed,target_rows,release_gate_rows,active_release_lift_rows,bc_after_accuracy,bc_after_loss,...,downtrend_accuracy,fold_gate,target_score,test_return,test_sharpe,test_max_dd,test_avg_pos,test_stress_avg_pos,test_buy_and_hold_return,test_relative_return
0,cap10_release_0.10,0.10,wf_2022,evaluated,20260526,505,40,0,0.790099,1.114388,...,0.970085,True,0.000000,5.156021,0.442735,7.908212,0.171053,0.122674,-76.614257,72.537447
1,cap10_release_0.10,0.10,wf_2023_2024,evaluated,20260527,756,40,0,0.667550,1.236124,...,0.968553,False,49.385503,20.503706,0.909436,12.291345,0.130533,0.081250,55.654474,-19.385503
2,cap10_release_0.15,0.15,wf_2022,evaluated,20260626,505,40,40,0.754455,1.148773,...,0.952991,True,0.000000,-5.195605,-0.297101,11.722326,0.156447,0.132558,-76.614257,71.918219
3,cap10_release_0.15,0.15,wf_2023_2024,evaluated,20260627,756,40,40,0.664901,1.240591,...,0.964361,False,50.403253,19.082635,0.859041,13.470361,0.129087,0.075329,55.654474,-20.384999
4,cap10_release_0.20,0.20,wf_2022,evaluated,20260726,505,40,40,0.774257,1.132945,...,0.944444,True,0.000000,-1.407176,-0.034873,11.401851,0.158553,0.129942,-76.614257,68.074895
5,cap10_release_0.20,0.20,wf_2023_2024,evaluated,20260727,756,40,40,0.606623,1.298624,...,0.872117,False,54.979558,26.064272,1.179569,8.226753,0.128237,0.082566,55.654474,-19.944298
6,cap10_release_0.25,0.25,wf_2022,evaluated,20260826,505,40,40,0.752475,1.157611,...,0.893162,False,47.224895,-23.381350,-1.251907,26.806224,0.162237,0.125000,-76.614257,55.207557
7,cap10_release_0.25,0.25,wf_2023_2024,evaluated,20260827,756,40,40,0.625166,1.281292,...,0.901468,False,54.287107,25.526909,1.163591,8.124669,0.129135,0.076316,55.654474,-19.269810
8,cap10_release_0.30,0.30,wf_2022,evaluated,20260926,505,40,40,0.756436,1.151666,...,0.884615,True,0.655263,-2.349035,-0.116438,9.261052,0.097237,0.048547,-76.614257,72.410846
9,cap10_release_0.30,0.30,wf_2023_2024,evaluated,20260927,756,40,40,0.621192,1.283379,...,0.893082,False,55.848306,20.503706,0.909436,12.291345,0.128682,0.080921,55.654474,-20.821938


In [6]:

evaluated = fold_results[fold_results['status'].eq('evaluated')].copy() if not fold_results.empty else pd.DataFrame()
strict_candidates = candidate_summary[candidate_summary['strict_gate'].astype(bool)].copy() if not candidate_summary.empty else pd.DataFrame()
passing_candidates = candidate_summary[candidate_summary['all_fold_gate'].astype(bool)].copy() if not candidate_summary.empty else pd.DataFrame()
if not strict_candidates.empty:
    selected = strict_candidates.sort_values(['mean_avg_pos', 'worst_max_dd'], ascending=[False, True]).iloc[0].to_dict()
    decision = 'cap10_release_balance_candidate_no_promotion'
    next_mechanism = 'run_fixed_split_wf2020_and_short_ppo_drift_screen_for_selected_release_cap'
elif not passing_candidates.empty:
    selected = passing_candidates.sort_values(['mean_target_score', 'worst_max_dd'], ascending=[True, True]).iloc[0].to_dict()
    decision = 'cap10_release_balance_underparticipation_no_promotion'
    next_mechanism = 'add_non_downtrend_rebound_participation_or_hysteresis_before_ppo'
else:
    selected = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
    decision = 'cap10_release_balance_failed_no_promotion'
    next_mechanism = 'diagnose_2023_2024_upside_capture_with_rebound_participation_before_ppo'

selected_release_cap = float(selected.get('release_cap', float('nan'))) if selected else float('nan')
selected_underparticipation = bool(selected and float(selected.get('mean_avg_pos', 0.0)) < MIN_MEAN_AVG_POS)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'sparse_release_drawdown_target_tightening_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'training_target_design', 'bc_sample_weighting'],
    'promotion_allowed': False,
    'promotion_blocker': 'This is a fold-local BC target-design screen only; no actor bundle is promoted or retained.',
    'risk_anchor_cap': RISK_ANCHOR_CAP,
    'release_cap_grid': RELEASE_CAP_GRID,
    'selected_release_cap': selected_release_cap if np.isfinite(selected_release_cap) else None,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'min_mean_avg_pos': MIN_MEAN_AVG_POS,
    'selected_underparticipation_warning': selected_underparticipation,
    'strict_candidate_count': int(len(strict_candidates)),
    'passing_candidate_count': int(len(passing_candidates)),
    'folds_evaluated': sorted(evaluated['fold'].unique().tolist()) if not evaluated.empty else [],
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'selected_targets_by_variant.csv',
        'target_variant_coverage.csv',
        'target_scope_audit.csv',
        'target_scope_summary.json',
        'target_lift_audit.csv',
        'target_lift_summary.json',
        'target_variant_summary.csv',
        'target_variant_fold_results.csv',
        'bc_train_metrics.csv',
        'bc_slice_metrics.csv',
        'action_distribution.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/1704.03732',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'coverage': coverage,
    'selected_candidate': selected,
    'candidate_summary': candidate_summary.to_dict(orient='records'),
    'fold_results': fold_results.to_dict(orient='records'),
    'target_variant_coverage': target_variant_coverage.to_dict(orient='records'),
    'source_context': {
        'experiment': 'sparse_rebound_release_target_revision_v1',
        'decision': source_selection.get('decision'),
        'conclusion': source_summary.get('conclusion', []),
    },
    'direct_comparison': {
        'experiment': 'sparse_release_drawdown_target_tightening_v1',
        'decision': direct_selection.get('decision'),
        'selected_target_cap': direct_selection.get('selected_target_cap'),
        'conclusion': direct_summary.get('conclusion', []),
    },
    'conclusion': [],
}
if candidate_summary.empty:
    summary['conclusion'].append('No release-cap variants were evaluated; do not continue this branch.')
else:
    for row in candidate_summary.sort_values('release_cap').to_dict(orient='records'):
        summary['conclusion'].append(
            f"release_cap={row['release_cap']:.2f}: strict={bool(row['strict_gate'])}, pass={bool(row['all_fold_gate'])}, "
            f"pass_count={int(row['fold_gate_pass_count'])}/2, slice_count={int(row['slice_gate_pass_count'])}/2, "
            f"worst MaxDD={row['worst_max_dd']:.2f}%, mean relative={row['mean_relative_return']:.2f}pp, "
            f"mean avg_pos={row['mean_avg_pos']:.3f}."
        )
    if not strict_candidates.empty:
        summary['conclusion'].append(
            f"Selected diagnostic release_cap={selected_release_cap:.2f}: it passes held-out return/drawdown, slice fit, and participation gates, but remains no-promotion until fixed-split, wf_2020, and PPO drift checks pass."
        )
    elif not passing_candidates.empty:
        summary['conclusion'].append(
            'At least one release-cap variant passes return/drawdown, but none clears the participation/slice gate combination; do not hand to PPO yet.'
        )
    else:
        summary['conclusion'].append(
            'No release-cap variant keeps both held-out folds inside +30pp relative and <=15% MaxDD. The stricter 2023-2024 window fails on upside capture versus buy-and-hold; loosening sparse release rows alone is not enough.'
        )
    if selected_underparticipation:
        summary['conclusion'].append(
            'Selected candidate is still below the mean exposure floor; next work should alter rebound participation semantics, not loosen drawdown caps blindly.'
        )

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
summary


{'experiment': 'sparse_release_cap10_rebound_balance_v1',
 'decision': 'cap10_release_balance_failed_no_promotion',
 'selection': {'experiment': 'sparse_release_cap10_rebound_balance_v1',
  'created_at': '2026-05-25T21:31:22.264945+00:00',
  'decision': 'cap10_release_balance_failed_no_promotion',
  'next_mechanism': 'diagnose_2023_2024_upside_capture_with_rebound_participation_before_ppo',
  'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
  'direct_comparison_baseline': 'sparse_release_drawdown_target_tightening_v1',
  'role': 'diagnose',
  'profile': 'sparse_release_bc_ppo_screen_v1',
  'changed_layer': ['model_selection',
   'training_target_design',
   'bc_sample_weighting'],
  'promotion_allowed': False,
  'promotion_blocker': 'This is a fold-local BC target-design screen only; no actor bundle is promoted or retained.',
  'risk_anchor_cap': 0.1,
  'release_cap_grid': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35],
  'selected_release_cap': 0.1,
  'target_relative_return_pp': 30.

In [7]:

lines = [
    f"## Runtime Conclusion: {EXPERIMENT}",
    "",
    f"- **Decision**: `{summary['decision']}`.",
    f"- **Promotion**: blocked. {selection['promotion_blocker']}",
    f"- **Selected release cap**: `{selection['selected_release_cap']}` with risk anchor `{selection['risk_anchor_cap']}`.",
    f"- **Strict candidates**: {selection['strict_candidate_count']} / {len(RELEASE_CAP_GRID)}.",
    f"- **Passing return/drawdown candidates**: {selection['passing_candidate_count']} / {len(RELEASE_CAP_GRID)}.",
    f"- **Underparticipation warning**: `{selection['selected_underparticipation_warning']}`.",
    "",
    "### Variant Evidence",
]
for row in candidate_summary.sort_values('release_cap').to_dict(orient='records'):
    lines.append(
        f"- `release_cap={row['release_cap']:.2f}`: strict={bool(row['strict_gate'])}, "
        f"pass={bool(row['all_fold_gate'])}, pass_count={int(row['fold_gate_pass_count'])}/2, "
        f"slice_count={int(row['slice_gate_pass_count'])}/2, worst MaxDD={row['worst_max_dd']:.2f}%, "
        f"mean relative={row['mean_relative_return']:.2f}pp, mean avg_pos={row['mean_avg_pos']:.3f}."
    )
lines += [
    "",
    "### Risk Notes",
    "- This is a BC target-design screen, not a runtime actor promotion.",
    "- The second fold tests 2023-2024 after target rows end in 2022, so it is stricter than the prior 2024-only cap screen.",
    "- A release cap that improves exposure but breaches <=15% MaxDD should not be handed to PPO; the previous PPO screen already showed drift risk.",
    "",
    "### Artifacts",
]
for sidecar in selection['sidecars']:
    lines.append(f"- `{sidecar}`")
lines += [
    "",
    "### Next Action",
    f"- `{selection['next_mechanism']}`.",
]
display(Markdown("\n".join(lines)))


## Runtime Conclusion: sparse_release_cap10_rebound_balance_v1

- **Decision**: `cap10_release_balance_failed_no_promotion`.
- **Promotion**: blocked. This is a fold-local BC target-design screen only; no actor bundle is promoted or retained.
- **Selected release cap**: `0.1` with risk anchor `0.1`.
- **Strict candidates**: 0 / 6.
- **Passing return/drawdown candidates**: 0 / 6.
- **Underparticipation warning**: `False`.

### Variant Evidence
- `release_cap=0.10`: strict=False, pass=False, pass_count=1/2, slice_count=2/2, worst MaxDD=12.29%, mean relative=26.58pp, mean avg_pos=0.151.
- `release_cap=0.15`: strict=False, pass=False, pass_count=1/2, slice_count=2/2, worst MaxDD=13.47%, mean relative=25.77pp, mean avg_pos=0.143.
- `release_cap=0.20`: strict=False, pass=False, pass_count=1/2, slice_count=1/2, worst MaxDD=11.40%, mean relative=24.07pp, mean avg_pos=0.143.
- `release_cap=0.25`: strict=False, pass=False, pass_count=0/2, slice_count=1/2, worst MaxDD=26.81%, mean relative=17.97pp, mean avg_pos=0.146.
- `release_cap=0.30`: strict=False, pass=False, pass_count=1/2, slice_count=1/2, worst MaxDD=12.29%, mean relative=25.79pp, mean avg_pos=0.113.
- `release_cap=0.35`: strict=False, pass=False, pass_count=1/2, slice_count=1/2, worst MaxDD=13.63%, mean relative=24.94pp, mean avg_pos=0.104.

### Risk Notes
- This is a BC target-design screen, not a runtime actor promotion.
- The second fold tests 2023-2024 after target rows end in 2022, so it is stricter than the prior 2024-only cap screen.
- A release cap that improves exposure but breaches <=15% MaxDD should not be handed to PPO; the previous PPO screen already showed drift risk.

### Artifacts
- `coverage.csv`
- `coverage.json`
- `selected_targets_by_variant.csv`
- `target_variant_coverage.csv`
- `target_scope_audit.csv`
- `target_scope_summary.json`
- `target_lift_audit.csv`
- `target_lift_summary.json`
- `target_variant_summary.csv`
- `target_variant_fold_results.csv`
- `bc_train_metrics.csv`
- `bc_slice_metrics.csv`
- `action_distribution.csv`
- `selection.json`
- `summary.json`

### Next Action
- `diagnose_2023_2024_upside_capture_with_rebound_participation_before_ppo`.